# Feature Engineering — Companion Notebook

> Companion for `src/my_mlops_project/pipelines/data_feat_engineering/`.
> Shows the engineered features, their relationship to the target, and the
> Hopsworks feature-store sync.

**Purpose:** derive predictive features over the four credit feature groups and
write them to the Hopsworks feature store (with a local-parquet canonical copy).

**Inputs:** `data/03_primary/cleaned_data.parquet`.
**Outputs:** `data/04_feature/feature_table.parquet`, plus 4 Hopsworks feature
groups and `data/08_reporting/feature_store_status.json`.

## Table of Contents
1. [Setup](#1-setup)
2. [Load cleaned data](#2-load-cleaned-data)
3. [Engineer features](#3-engineer-features)
4. [Explore: do the features separate defaulters?](#4-explore-do-the-features-separate-defaulters)
5. [The feature store (write + read-back)](#5-the-feature-store-write--read-back)
6. [Validate output](#6-validate-output)
7. [Notes for the report](#7-notes-for-the-report)

## 1. Setup

In [1]:
import sys, json
from pathlib import Path
import yaml
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
DATA = PROJECT_ROOT / "data"

params = yaml.safe_load(open(PROJECT_ROOT / "conf" / "base" / "parameters.yml"))["data_feat_engineering"]
params

{'primary_key': 'customer_id',
 'event_time': 'event_time',
 'feature_group_version': 1,
 'feature_groups': {'credit_demographic': ['LIMIT_BAL',
   'SEX',
   'EDUCATION',
   'MARRIAGE',
   'AGE',
   'default'],
  'credit_payment_status': ['PAY_0',
   'PAY_2',
   'PAY_3',
   'PAY_4',
   'PAY_5',
   'PAY_6',
   'pay_delay_max',
   'pay_delay_mean',
   'n_months_delayed'],
  'credit_bill': ['BILL_AMT1',
   'BILL_AMT2',
   'BILL_AMT3',
   'BILL_AMT4',
   'BILL_AMT5',
   'BILL_AMT6',
   'avg_bill',
   'max_bill',
   'bill_trend',
   'util_ratio'],
  'credit_payment_amt': ['PAY_AMT1',
   'PAY_AMT2',
   'PAY_AMT3',
   'PAY_AMT4',
   'PAY_AMT5',
   'PAY_AMT6',
   'avg_pay_amt',
   'total_pay',
   'pct_paid']}}

## 2. Load cleaned data

In [2]:
cleaned = pd.read_parquet(DATA / "03_primary" / "cleaned_data.parquet")
print("cleaned shape:", cleaned.shape)
print((cleaned.head()).to_string())

cleaned shape: (29965, 24)
   LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  PAY_5  PAY_6  BILL_AMT1  BILL_AMT2  BILL_AMT3  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  PAY_AMT4  PAY_AMT5  PAY_AMT6  default
0      20000    2          2         1   24      2      2     -1     -1     -2     -2       3913       3102        689          0          0          0         0       689         0         0         0         0        1
1     120000    2          2         2   26     -1      2      0      0      0      2       2682       1725       2682       3272       3455       3261         0      1000      1000      1000         0      2000        1
2      90000    2          2         2   34      0      0      0      0      0      0      29239      14027      13559      14331      14948      15549      1518      1500      1000      1000      1000      5000        0
3      50000    2          2         1   37      0      0      0      0      0      0    

## 3. Engineer features

We call the production node so the notebook and pipeline cannot drift. The new
features summarise the six-month history into single signals a model (and SHAP)
can use: utilisation, delay counts, bill trend, repayment ratio.

In [3]:
from my_mlops_project.pipelines.data_feat_engineering.nodes import engineer_features

feature_table = engineer_features(cleaned, params)
new_cols = [c for c in feature_table.columns if c not in cleaned.columns]
print("new features:", new_cols)
print((feature_table[["customer_id"] + new_cols[1:-1]].head()).to_string())

engineer_features: 24 -> 36 columns (29965 rows)
new features: ['customer_id', 'pay_delay_max', 'pay_delay_mean', 'n_months_delayed', 'avg_bill', 'max_bill', 'bill_trend', 'avg_pay_amt', 'total_pay', 'util_ratio', 'pct_paid', 'event_time']
   customer_id  pay_delay_max  pay_delay_mean  n_months_delayed      avg_bill  max_bill  bill_trend  avg_pay_amt  total_pay  util_ratio  pct_paid
0            0              2       -0.333333                 2   1284.000000      3913        3913   114.833333        689    0.064200  0.089434
1            1              2        0.500000                 2   2846.166667      3455        -579   833.333333       5000    0.023718  0.292791
2            2              0        0.000000                 0  16942.166667     29239       13690  1836.333333      11018    0.188246  0.108388
3            3              0        0.000000                 0  38555.666667     49291       17443  1398.000000       8388    0.771113  0.036259
4            4              0 

## 4. Explore: do the features separate defaulters?

A quick check of whether the engineered features actually carry signal — we
compare their average for defaulters (`default=1`) vs non-defaulters (`default=0`).
A large gap means the feature is predictive (a preview of the SHAP story later).

In [4]:
signal = feature_table.groupby("default")[
    ["util_ratio", "pay_delay_max", "n_months_delayed", "pct_paid", "avg_bill"]
].mean().T
signal["gap"] = (signal[1] - signal[0])
print((signal.round(3)).to_string())

default                   0          1       gap
util_ratio            0.352      0.450     0.098
pay_delay_max         0.202      1.275     1.073
n_months_delayed      0.504      1.996     1.492
pct_paid              0.372      0.251    -0.120
avg_bill          45461.122  43509.584 -1951.538


In [5]:
# Correlation of each engineered feature with the target (quick ranking).
eng = ["util_ratio", "pay_delay_max", "pay_delay_mean", "n_months_delayed",
       "avg_bill", "max_bill", "bill_trend", "avg_pay_amt", "total_pay", "pct_paid"]
feature_table[eng + ["default"]].corr()["default"].drop("default").sort_values(key=abs, ascending=False).round(3)

n_months_delayed    0.398
pay_delay_max       0.331
pay_delay_mean      0.282
util_ratio          0.115
pct_paid           -0.111
total_pay          -0.102
avg_pay_amt        -0.102
max_bill           -0.041
bill_trend         -0.026
avg_bill           -0.013
Name: default, dtype: float64

## 5. The feature store (write + read-back)

The pipeline writes **four feature groups** to Hopsworks — `credit_demographic`,
`credit_payment_status`, `credit_bill`, `credit_payment_amt` — each keyed by
`customer_id` with an `event_time`, then reads one back to prove the round-trip.

Design choice (robustness): the **local** `feature_table.parquet` is the
canonical input for modelling, so the pipeline reproduces even if Hopsworks is
unavailable. The `sync_to_feature_store` node is best-effort and records what
happened in `feature_store_status.json`.

In [6]:
# The result of the last pipeline run's Hopsworks sync.
status_path = DATA / "08_reporting" / "feature_store_status.json"
if status_path.exists():
    status = json.load(open(status_path))
    print(json.dumps(status, indent=2))
else:
    print("Run `kedro run --pipeline=data_feat_engineering` first to produce the status.")

{
  "connected": true,
  "feature_groups_written": [
    "credit_demographic",
    "credit_payment_status",
    "credit_bill",
    "credit_payment_amt"
  ],
  "rows_written": 29965,
  "read_back_rows": 29965,
  "note": "Feature store write + read-back OK (online store).",
  "storage": "online"
}


**Reading features back** (reference) — how a downstream consumer would pull a
feature group or build a feature view from Hopsworks:

```python
import hopsworks
project = hopsworks.login(api_key_value=API_KEY, project="MLOps2026")
fs = project.get_feature_store()

# read one group back
demo = fs.get_feature_group("credit_demographic", version=1)
df = demo.read()

# or join all four into a feature view + training data (avoids train/serve skew)
fv = fs.get_or_create_feature_view(name="credit_view", version=1,
        query=demo.select_all(), labels=["default"])
X, y = fv.training_data()
```

## 6. Validate output

In [7]:
assert feature_table["customer_id"].is_unique, "customer_id must be unique"
assert feature_table["util_ratio"].notna().all(), "util_ratio has NaNs"
assert "event_time" in feature_table.columns, "event_time required by the feature store"
print("Feature table OK:", feature_table.shape)

Feature table OK: (29965, 36)


## 7. Notes for the report

> For [`../report/REPORT_OUTLINE.md`](../report/REPORT_OUTLINE.md) §3 (features) and §4 (production).

- **Engineered features** turn the 6-month history into single signals:
  `util_ratio` (credit utilisation), `pay_delay_max` / `n_months_delayed`
  (arrears), `bill_trend`, `pct_paid` (repayment ratio). The defaulter-vs-non
  gap in §4 shows arrears-related features carry the strongest signal.
- **Feature store:** four feature groups written to Hopsworks (`MLOps2026`) and
  read back — both **write and read**, not just write.
- **Production risk / mitigation:** the feature store is a *managed* dependency
  (the cluster was mid-upgrade during development). We mitigate by keeping the
  local `feature_table.parquet` canonical, so the pipeline reproduces even when
  Hopsworks is unavailable — the `feature_store_status.json` records each run's
  outcome.